# Analysis — `n1m5_T15_obs5000_seed42`

Compare the **credulous** vs **vigilant** listener under two speakers (informative, persuasive), across all 9 true thetas. Each figure is **2 rows × 9 columns**:
- Row 1: speaker is `inf` (informative).
- Row 2: speaker is `persp` (persuasive, `pers+`).
- Column k: true θ = 0.1·k.
- x = round (0..15), y = some summary of the listener's posterior. Two lines + 95% CI shading: blue = credulous, orange = vigilant. Round 0 is the (flat) prior; recorded round t=0 is plotted as round 1.

Three figures, differing only in the per-trajectory quantity and the cross-trajectory aggregator:

| figure | per-trajectory quantity | cross-trajectory aggregator |
|---|---|---|
| 1 | $\mathbb{E}[\theta \mid u_{0..t}]$ | **median** |
| 2 | $\mathbb{E}[\theta \mid u_{0..t}]$ | **mean** |
| 3 | posterior **median** of θ | median |

The notebook is factorized: **Load → Compute → Aggregate → Visualize**.

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Bootstrap repo root onto sys.path so absolute imports work from a notebook.
HERE = Path.cwd().resolve()
# analyze.ipynb -> n1m5.../ -> simulation_experiments/ -> simulations/ -> models/ -> repo root
REPO_ROOT = HERE.parents[3]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from models.simulations.simulation_experiments.n1m5_T15_obs5000_seed42.io import load_beliefs

DATA_ROOT = HERE / 'raw_do_not_track'
print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT.is_dir(), f'expected experiment data at {DATA_ROOT}'

In [ ]:
# Identifiers — must match the directory names produced by run.py.
SPEAKERS = {
    'inf':   'inf_L1strat_a3_b1_uiF',
    'persp': 'persp_L1strat_a3_b0_uiF',
}
LISTENERS = {
    'credulous': 'credulous_L1coop_a3_uiF',
    'vigilant':  'vigilant_L1strat_a3_uiF',
}

THETAS = [round(0.1 * k, 1) for k in range(1, 10)]   # [0.1, ..., 0.9]

# Plot config
LISTENER_COLORS = {'credulous': 'tab:blue', 'vigilant': 'tab:orange'}
SPEAKER_TITLES = {'inf': 'Informative speaker', 'persp': 'Persuasive (pers+) speaker'}

## 1. Load — read belief Datasets for the (speaker × listener) cells we care about

In [ ]:
def load_belief_grid(data_root, speakers, listeners):
    """Return a dict {(spk_key, lst_key): xr.Dataset} for every (speaker, listener) pair."""
    out = {}
    for spk_key, spk_dir in speakers.items():
        for lst_key, lst_dir in listeners.items():
            path = data_root / spk_dir / lst_dir
            out[(spk_key, lst_key)] = load_beliefs(path)
    return out

beliefs = load_belief_grid(DATA_ROOT, SPEAKERS, LISTENERS)
for k, ds in beliefs.items():
    print(f'{k}: sizes={dict(ds.sizes)} path={ds.attrs["execution_path"]}')

## 2. Compute — per-trajectory summaries, with the flat prior prepended at round 0

Two per-trajectory quantities, each as a numpy array of shape `(n_theta_true, n_traj, n_rounds)`:

- **Expected theta**: $\mathbb{E}[\theta \mid u_{0..t}] = \sum_\theta \theta \cdot P(\theta \mid u_{0..t})$.
- **Posterior median**: the value $\theta^\star$ at which the posterior CDF crosses 0.5, with linear interpolation between adjacent grid points.

For a uniform prior over θ ∈ {0.0, 0.1, …, 1.0}, both summaries equal 0.5 at round 0, which is what we prepend.

We collapse `obs_idx` × `utt_idx` into a single trajectory axis so downstream aggregation is straightforward; with `n_utt_seq=1` for this experiment the two axes carry the same information.

In [ ]:
def expected_theta_with_prior(belief_ds):
    """E[theta | u_{0..t}] per trajectory, with a synthetic round 0 = prior mean prepended."""
    e = (belief_ds.belief_theta * belief_ds.theta).sum('theta')
    e = e.stack(traj=('obs_idx', 'utt_idx')).transpose('theta_true', 'traj', 't')
    arr = e.values

    prior_value = float(belief_ds.theta.mean().item())     # uniform-grid mean
    n_thetas, n_traj, _ = arr.shape
    prior_block = np.full((n_thetas, n_traj, 1), prior_value, dtype=arr.dtype)
    return np.concatenate([prior_block, arr], axis=2)


def _posterior_median_along_last(probs, grid):
    """
    Median of a discrete distribution via linear CDF interpolation.
    probs: shape (..., n_grid), normalized along last axis.
    grid:  1-D ascending array of length n_grid.
    Returns: shape (...,) median value.
    """
    cdf = np.cumsum(probs, axis=-1)
    idx = (cdf >= 0.5).argmax(axis=-1)                      # first crossing
    idx_lo = np.clip(idx - 1, 0, None)
    cdf_hi = np.take_along_axis(cdf, idx[..., None], axis=-1).squeeze(-1)
    cdf_lo = np.where(idx == 0, 0.0,
                      np.take_along_axis(cdf, idx_lo[..., None], axis=-1).squeeze(-1))
    p = cdf_hi - cdf_lo
    frac = np.where(p > 1e-15, (0.5 - cdf_lo) / np.maximum(p, 1e-15), 0.0)
    frac = np.clip(frac, 0.0, 1.0)
    theta_hi = grid[idx]
    theta_lo = np.where(idx == 0, grid[0], grid[idx_lo])
    return theta_lo + frac * (theta_hi - theta_lo)


def posterior_median_with_prior(belief_ds):
    """Posterior median(θ) per trajectory + synthetic round-0 prior median."""
    grid = belief_ds.theta.values
    bt = belief_ds.belief_theta.stack(traj=('obs_idx', 'utt_idx')).transpose(
        'theta_true', 'traj', 't', 'theta'
    )
    arr = bt.values                                         # (theta_true, traj, t, theta)
    med = _posterior_median_along_last(arr, grid)           # (theta_true, traj, t)

    prior_med = float(_posterior_median_along_last(
        np.full(grid.shape, 1.0 / len(grid)), grid))
    n_thetas, n_traj, _ = med.shape
    prior_block = np.full((n_thetas, n_traj, 1), prior_med, dtype=med.dtype)
    return np.concatenate([prior_block, med], axis=2)


e_theta = {k: expected_theta_with_prior(ds) for k, ds in beliefs.items()}
post_med = {k: posterior_median_with_prior(ds) for k, ds in beliefs.items()}

for k in beliefs:
    print(
        f'{k}: e_theta {e_theta[k].shape} '
        f'in [{e_theta[k].min():.3f}, {e_theta[k].max():.3f}]; '
        f'post_med {post_med[k].shape} '
        f'in [{post_med[k].min():.3f}, {post_med[k].max():.3f}]'
    )

## 3. Aggregate — central tendency + 95% interval across trajectories

`summarize_traj` is parameterized by the central-tendency statistic (`'median'` or `'mean'`). The 95% interval (2.5 / 97.5 percentiles) is the same across choices — it's just a quantile of the trajectory distribution.

In [ ]:
def summarize_traj(arr, center='median', lo_pct=2.5, hi_pct=97.5):
    """arr: (theta_true, traj, round). Returns (central, lo, hi), each (theta_true, round)."""
    if center == 'median':
        central = np.median(arr, axis=1)
    elif center == 'mean':
        central = np.mean(arr, axis=1)
    else:
        raise ValueError(f"center must be 'median' or 'mean', got {center!r}")
    lo = np.percentile(arr, lo_pct, axis=1)
    hi = np.percentile(arr, hi_pct, axis=1)
    return central, lo, hi

summaries_e_theta_med  = {k: summarize_traj(arr, center='median') for k, arr in e_theta.items()}
summaries_e_theta_mean = {k: summarize_traj(arr, center='mean')   for k, arr in e_theta.items()}
summaries_post_med     = {k: summarize_traj(arr, center='median') for k, arr in post_med.items()}

# Sanity check at theta_true=0.5 (idx 4), final round
print('Persuasive speaker, theta_true=0.5, last round:')
for label, summ in [('median of E[θ]',           summaries_e_theta_med),
                     ('mean of E[θ]',             summaries_e_theta_mean),
                     ('median of posterior med', summaries_post_med)]:
    for lst in ['credulous', 'vigilant']:
        c, lo, hi = summ[('persp', lst)]
        print(f'  {label:24s} {lst:10s} central={c[4, -1]:.3f} '
              f'95%CI=[{lo[4, -1]:.3f}, {hi[4, -1]:.3f}]')

## 4. Visualize — three figures, same shape, different summary

In [ ]:
def plot_belief_grid(summaries, thetas, speakers, listeners,
                     listener_colors, speaker_titles,
                     y_label, fig_title=None):
    n_rows = len(speakers)
    n_cols = len(thetas)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True,
        squeeze=False,
    )

    any_central = next(iter(summaries.values()))[0]
    n_rounds = any_central.shape[1]
    rounds = np.arange(n_rounds)

    for row, spk_key in enumerate(speakers):
        for col, theta_true in enumerate(thetas):
            ax = axes[row, col]
            ax.axhline(theta_true, color='black', linestyle='--',
                       linewidth=0.8, alpha=0.6)

            for lst_key in listeners:
                central, lo, hi = summaries[(spk_key, lst_key)]
                color = listener_colors[lst_key]
                ax.plot(rounds, central[col], color=color, label=lst_key, linewidth=1.6)
                ax.fill_between(rounds, lo[col], hi[col], color=color, alpha=0.18)

            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk_key]}\n{y_label}', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')
            ax.set_ylim(0, 1)
            ax.set_xlim(0, n_rounds - 1)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if fig_title is not None:
        fig.suptitle(fig_title, fontsize=12, y=1.02)
    fig.legend(handles, labels, loc='upper center',
               ncol=len(labels), bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig

### Figure 1 — median of $\mathbb{E}[\theta \mid u_{0..t}]$ across trajectories

In [ ]:
plot_belief_grid(
    summaries_e_theta_med, THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label='median $\\mathbb{E}[\\theta \\mid u_{0..t}]$',
    fig_title='Median of expected θ across trajectories (95% CI shading)',
)
plt.show()

### Figure 2 — mean of $\mathbb{E}[\theta \mid u_{0..t}]$ across trajectories

In [ ]:
plot_belief_grid(
    summaries_e_theta_mean, THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label='mean $\\mathbb{E}[\\theta \\mid u_{0..t}]$',
    fig_title='Mean of expected θ across trajectories (95% CI shading)',
)
plt.show()

### Figure 3 — median of posterior-median θ across trajectories

For each trajectory and round, take the **median value of θ under the listener's posterior** (CDF crossing 0.5, with linear interpolation between grid points). Then take the median of those across trajectories.

In [ ]:
plot_belief_grid(
    summaries_post_med, THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label=r'median $\theta^\star$ (posterior med)',
    fig_title='Median of posterior median θ across trajectories (95% CI shading)',
)
plt.show()

## What to look for

- **Truth line** (black dashed) marks $\theta_{true}$ for each column.
- Under the **informative speaker** (row 1), both listeners should track the truth — credulous is *correctly* assuming the speaker is informative.
- Under the **persuasive speaker** (row 2), credulous should be **biased**: it still assumes informativeness, so persuasive utterances move its summary up. The vigilant listener corrects for this.
- **Median vs mean of $\mathbb{E}[\theta]$** (figs 1 vs 2): mean is sensitive to outlier trajectories where the posterior collapses to a wrong corner; median is robust. If the two diverge meaningfully, the trajectory distribution is skewed.
- **Posterior median (fig 3)** is bounded by the θ grid by construction (interpolated, but anchored at 0..1), unlike $\mathbb{E}[\theta]$ which is a weighted average. For unimodal posteriors near a single grid point, posterior median ≈ MAP.